# Beijing PM2.5 Forecasting  
## Notebook 07 — Tree-Based Models (Random Forest & Boosting)

In this notebook, we evaluate non-linear, tree-based models to improve upon the linear regression baseline.

Tree-based models can:
- capture non-linear relationships
- handle feature interactions automatically
- better model extreme pollution events

In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load Time-Based Splits

We reuse the chronological splits created earlier to ensure a fair comparison with baseline models.

In [3]:
X_train = pd.read_csv("../data/processed/X_train.csv", index_col="datetime", parse_dates=True)
y_train = pd.read_csv("../data/processed/y_train.csv", index_col="datetime", parse_dates=True).squeeze()

X_val = pd.read_csv("../data/processed/X_val.csv", index_col="datetime", parse_dates=True)
y_val = pd.read_csv("../data/processed/y_val.csv", index_col="datetime", parse_dates=True).squeeze()

X_test = pd.read_csv("../data/processed/X_test.csv", index_col="datetime", parse_dates=True)
y_test = pd.read_csv("../data/processed/y_test.csv", index_col="datetime", parse_dates=True).squeeze()

## 2. Evaluation Metrics

In [4]:
def evaluate_model(y_true, y_pred, name="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"{name} Performance:")
    print(f"  MAE : {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²  : {r2:.3f}")
    print("-" * 30)

## 3. Random Forest Regressor

Random Forest builds an ensemble of decision trees and averages their predictions, allowing it to capture complex, non-linear patterns.

In [5]:
# Train the random forest regressor
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_val_pred_rf = rf.predict(X_val)

evaluate_model(y_val, y_val_pred_rf, "Random Forest (Validation)")

Random Forest (Validation) Performance:
  MAE : 3.74
  RMSE: 10.03
  R²  : 0.979
------------------------------


In [6]:
# Test evaluation
y_test_pred_rf = rf.predict(X_test)
evaluate_model(y_test, y_test_pred_rf, "Random Forest (Test)")

Random Forest (Test) Performance:
  MAE : 3.76
  RMSE: 7.78
  R²  : 0.993
------------------------------


## 4. Gradient Boosting Regressor

Gradient Boosting is a boosting-based tree model similar in spirit to XGBoost, but available directly in scikit-learn.

In [7]:
# Train gradient boosting
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

gbr.fit(X_train, y_train)

y_val_pred_gbr = gbr.predict(X_val)

evaluate_model(y_val, y_val_pred_gbr, "Gradient Boosting (Validation)")

Gradient Boosting (Validation) Performance:
  MAE : 4.89
  RMSE: 10.09
  R²  : 0.979
------------------------------


In [8]:
y_test_pred_gbr = gbr.predict(X_test)
evaluate_model(y_test, y_test_pred_gbr, "Gradient Boosting (Test)")

Gradient Boosting (Test) Performance:
  MAE : 5.07
  RMSE: 8.96
  R²  : 0.991
------------------------------


In [14]:
# View results in a nice table
results_tree = pd.DataFrame({
    "Model": [
        "Random Forest (Validation)",
        "Random Forest (Test)",
        "Gradient Boosting (Validation)",
        "Gradient Boosting (Test)"
    ],
    "MAE": [
        mean_absolute_error(y_val, y_val_pred_rf),
        mean_absolute_error(y_test, y_test_pred_rf),
        mean_absolute_error(y_val, y_val_pred_gbr),
        mean_absolute_error(y_test, y_test_pred_gbr)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, y_val_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_test_pred_rf)),
        np.sqrt(mean_squared_error(y_val, y_val_pred_gbr)),
        np.sqrt(mean_squared_error(y_test, y_test_pred_gbr))
    ],
    "R2": [
        r2_score(y_val, y_val_pred_rf),
        r2_score(y_test, y_test_pred_rf),
        r2_score(y_val, y_val_pred_gbr),
        r2_score(y_test, y_test_pred_gbr)
    ]
})

results_tree.round(3)

,Model,MAE,RMSE,R2
0,Random Forest (Validation),3.736,10.030,0.979
1,Random Forest (Test),3.760,7.781,0.993
2,Gradient Boosting (Validation),4.892,10.086,0.979
3,Gradient Boosting (Test),5.075,8.963,0.991


## Model Performance Interpretation

Both tree-based models significantly outperform the linear baseline, confirming that PM2.5 dynamics are highly non-linear.

### Key observations:
- **Random Forest** achieves the best overall performance, with:
  - Lowest MAE on both validation and test sets
  - Highest R² on the test set (0.993)
- **Gradient Boosting** performs slightly worse but still generalizes well, indicating stable learning without severe overfitting.
- The small gap between validation and test metrics suggests **strong generalization** for both models.

### Conclusion:
Random Forest is selected as the **primary production model**, while Gradient Boosting serves as a strong secondary benchmark.


In [15]:
# Save models
import joblib
import os

# Create model directory
os.makedirs("../models", exist_ok=True)

# Save Random Forest model
joblib.dump(rf, "../models/random_forest_pm25.pkl")

# Save Gradient Boosting model (XGBoost-style)
joblib.dump(gbr, "../models/gradient_boosting_pm25.pkl")

print("Models saved successfully.")

Models saved successfully.
